# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmeer-Siddiqui125/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a transparent baseline for the Search Intelligence classification lane. The rule uses only signals available at the **2026-03-15 decision moment**. The future outcome window is used only to audit the signals, not as an input to the score.

The baseline is decision-support, not causal proof. Its purpose is to create a simple benchmark that the Week-5 model must beat.

## Setup

Week 3 defined the honest frame: features come from **2026-03-01 through 2026-03-15**, while the future engagement label comes from **2026-03-16 through 2026-03-31**. The label is 1 when future engagement rate is at or above the median positive future engagement rate, otherwise 0.

The baseline itself will not use `future_engagement`, `future_engagement_rate`, or any other future-period field.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(
    "CREATE SECRET hf_secret (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print('DuckDB connected to Hugging Face')

## 1. Signal checks before the rule

I am checking two signals that the rule will use:

1. `gsc_impressions` — linked to the quick-win idea because search visibility creates an observable opportunity to review content.
2. `gsc_avg_position` — linked to the CTR-fix/search-position logic because position gives context for interpreting search performance.

The bucket tables below show the number of observations (`n`) and the observed future-engagement positive rate. The verdict is based on the direction of the bucket pattern, not on causal interpretation.

In [ ]:
# Build the honest March decision frame.
# Features: March 1-15. Label: March 16-31.

honest_frame = con.execute(f"""
WITH features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS gsc_impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS gsc_clicks,
        AVG(CASE WHEN gsc_data_available IS TRUE AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS gsc_avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM read_parquet('{path}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),
future_content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS future_sessions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS future_engaged_sessions
    FROM read_parquet('{path}')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
),
labeled AS (
    SELECT
        client_hash_id,
        content_hash_id,
        future_engaged_sessions * 1.0 / future_sessions AS future_engagement_rate
    FROM future_content
    WHERE future_sessions > 0
),
threshold AS (
    SELECT MEDIAN(future_engagement_rate) AS median_rate
    FROM labeled
    WHERE future_engagement_rate > 0
)
SELECT
    f.*,
    l.future_engagement_rate,
    CASE
        WHEN l.future_engagement_rate > 0
             AND l.future_engagement_rate >= t.median_rate THEN 1
        ELSE 0
    END AS future_engagement
FROM features f
INNER JOIN labeled l
    ON f.client_hash_id = l.client_hash_id
   AND f.content_hash_id = l.content_hash_id
CROSS JOIN threshold t
""").df()

print('Honest frame shape:', honest_frame.shape)
print('Label distribution:')
print(honest_frame['future_engagement'].value_counts().sort_index())

In [ ]:
# Signal 1: gsc_impressions

signal1 = honest_frame.copy()
signal1['impression_bucket'] = pd.cut(
    signal1['gsc_impressions'],
    bins=[-1, 100, 500, 2000, np.inf],
    labels=['0-100', '101-500', '501-2000', '2000+']
)

impression_table = (
    signal1.groupby('impression_bucket', observed=False)
    .agg(n=('future_engagement', 'size'), positive_rate=('future_engagement', 'mean'))
    .reset_index()
)
print('Signal 1: gsc_impressions')
display(impression_table)

rates = impression_table['positive_rate'].dropna().tolist()
if len(rates) >= 2 and all(rates[i] <= rates[i+1] for i in range(len(rates)-1)) and rates[-1] > rates[0]:
    impression_verdict = 'CONFIRMED'
elif len(rates) >= 2 and all(rates[i] >= rates[i+1] for i in range(len(rates)-1)) and rates[-1] < rates[0]:
    impression_verdict = 'OPPOSITE'
else:
    impression_verdict = 'MIXED'

print('Verdict:', impression_verdict)

In [ ]:
# Signal 2: gsc_avg_position
# Position 0 is treated as missing/no valid position, not as rank zero.

signal2 = honest_frame[honest_frame['gsc_avg_position'].notna()].copy()
signal2['position_bucket'] = pd.cut(
    signal2['gsc_avg_position'],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=['1-3', '4-10', '11-20', '21-50', '50+']
)

position_table = (
    signal2.groupby('position_bucket', observed=False)
    .agg(n=('future_engagement', 'size'), positive_rate=('future_engagement', 'mean'))
    .reset_index()
)
print('Signal 2: gsc_avg_position')
display(position_table)

rates = position_table['positive_rate'].dropna().tolist()
if len(rates) >= 2 and all(rates[i] >= rates[i+1] for i in range(len(rates)-1)) and rates[0] > rates[-1]:
    position_verdict = 'CONFIRMED'
elif len(rates) >= 2 and all(rates[i] <= rates[i+1] for i in range(len(rates)-1)) and rates[0] < rates[-1]:
    position_verdict = 'OPPOSITE'
else:
    position_verdict = 'MIXED'

print('Verdict:', position_verdict)

### Signal interpretation

The verdicts above are descriptive. `CONFIRMED` means the observed bucket pattern moves in the expected direction; `OPPOSITE` means it moves in the opposite direction; `MIXED` means the pattern is not consistently monotonic. A clearly negative result is useful because it can prevent a weak signal from becoming a rule input.

## 2. Encode ONE rule and build the ranked queue

### Rule

Prioritize content that has meaningful search visibility and is sitting in a reviewable search-position range.

**Score:**
- +2 points when `gsc_impressions >= 500`.
- +1 point when `gsc_avg_position` is between 11 and 50.

**One reason code:** `visible_needs_position_review`

**Action label:** `review_search_performance`

The thresholds are intentionally simple and hand-written. They are a baseline, not learned parameters. The score uses only decision-time signals.

In [ ]:
# Build the queue from decision-time features only.
queue = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS gsc_impressions,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS gsc_clicks,
    AVG(CASE WHEN gsc_data_available IS TRUE AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS gsc_avg_position,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
FROM read_parquet('{path}')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
GROUP BY client_hash_id, content_hash_id
""").df()

queue['score'] = (
    (queue['gsc_impressions'] >= 500).astype(int) * 2
    + queue['gsc_avg_position'].between(11, 50, inclusive='both').fillna(False).astype(int)
)

queue['reason_code'] = np.where(
    queue['score'] > 0,
    'visible_needs_position_review',
    'not_prioritized'
)

queue['action'] = np.where(
    queue['score'] > 0,
    'review_search_performance',
    'monitor'
)

queue = queue.sort_values(
    ['score', 'gsc_impressions'],
    ascending=[False, False]
).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue) + 1)

output_cols = [
    'rank', 'client_hash_id', 'content_hash_id',
    'gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
    'score', 'reason_code', 'action'
]

import os
os.makedirs('work/outputs', exist_ok=True)
queue[output_cols].to_csv(
    'work/outputs/baseline_action_score.csv',
    index=False
)

print('Rows ranked:', len(queue))
print('CSV written: work/outputs/baseline_action_score.csv')
display(queue.head(10)[output_cols])

## 3. Top-10 review

For each of the ten highest-ranked rows, record the action, why it is there, and what could make the pick wrong. This is a skeptical review of the rule rather than a claim that the rule is correct.

In [ ]:
top10 = queue.head(10).copy()

review_rows = []
for _, row in top10.iterrows():
    why = (
        f"{row['gsc_impressions']:.0f} impressions and average position "
        f"{row['gsc_avg_position']:.1f} meet the baseline's visible/reviewable conditions."
        if pd.notna(row['gsc_avg_position'])
        else f"{row['gsc_impressions']:.0f} impressions meet the visibility condition, but position is missing."
    )
    wrong = (
        "The impressions may come from low-value queries, or the position may not represent a stable/actionable opportunity."
    )
    review_rows.append({
        'rank': int(row['rank']),
        'action': row['action'],
        'why_it_is_here': why,
        'what_would_make_it_wrong': wrong
    })

review = pd.DataFrame(review_rows)
display(review)

## 4. Weak picks + leakage check

Weak picks are items that receive a non-zero score but may still be poor recommendations because the rule cannot see query quality, content quality, business value, or stability of the observed position.

The queue does not use `future_engagement`, `future_engagement_rate`, future-period performance, trend-derived outcomes, product flags, or identifiers as predictive measurements. Those fields are excluded from the score.

In [ ]:
print('Leakage check')
print('- Decision-time inputs:', ['gsc_impressions', 'gsc_avg_position'])
print('- Future outcome used in score:', False)
print('- Label-derived input used in score:', False)
print('- Product flag used in score:', False)
print('- Future window in score:', False)

weak_picks = queue[queue['score'] > 0].tail(5)[output_cols]
print('\nExample weak/low-priority scored picks:')
display(weak_picks)

## Self-check

- [ ] Both signal bucket tables are visible and include `n`.
- [ ] Both signals have one-word verdicts: CONFIRMED, OPPOSITE, MIXED, or FALSE.
- [ ] At least one signal is linked to a real FlyRank flag idea.
- [ ] The rule has one score, one reason code, and one action label.
- [ ] `work/outputs/baseline_action_score.csv` is generated by this notebook.
- [ ] Ten rows have an action, why they are there, and what would make them wrong.
- [ ] No future-window or label-derived field is used by the score.
- [ ] The notebook runs top to bottom with no errors.
- [ ] The notebook is executed and committed under `work/notebooks/w04_baseline_score.ipynb`.

**Note:** the CSV stays out of git by design. Commit the notebook and any permitted metrics/figure receipts, not the generated data file.